# Rapport Technique : Comment trouver des conditions initiales permettant la stabilité du problème à 3 corps ?

## 1. Introduction et Contexte Historique

### La Problématique de Stabilité
Contrairement au problème à 2 corps (résolu par Newton), le système à 3 corps est intrinsèquement instable et chaotique dans la majorité des configurations. Trouver des orbites stables nécessite soit une hiérarchie de masses et de distances, soit des conditions initiales extrêmement précises pour des solutions périodiques particulières.

### Rappel Historique
Le problème à 3 corps a marqué l'histoire des sciences. Pour ne citer que quelques dates importantes :
*   **Isaac Newton (XVIIe siècle)** : Pose les bases avec la loi de la gravitation, mais échoue à trouver une solution générale pour la Lune (perturbée par le Soleil et la Terre).
*   **Euler et Lagrange (XVIIIe siècle)** : Découvrent les premières solutions particulières (points de Lagrange, configurations alignées ou triangulaires).
*   **Henri Poincaré (Fin XIXe siècle)** : Démontre que le problème n'est pas intégrable et met en évidence la sensibilité aux conditions initiales, posant les jalons de la **théorie du chaos**.

## 2. Facteurs de Stabilité du Système

Pour qu'un système à 3 corps soit stable sur le long terme, plusieurs facteurs entrent en jeu :

### Hiérarchie des Masses
C'est le cas le plus courant dans la nature (ex: Soleil-Terre-Lune). Un corps lourd central domine le potentiel gravitationnel, tandis que les autres corps sont beaucoup plus légers et gravitent à des distances suffisamment éloignées pour ne pas se perturber de manière catastrophique.

### Solutions Périodiques et Figures Théoriques
Certaines configurations spécifiques, bien que rares, permettent une stabilité dynamique :
*   **La Figure en 8** : Découverte par Chenciner et Montgomery en 2000, où trois corps de masses égales se poursuivent sur une trajectoire en forme de huit.
*   **Le Triangle Isocèle** : Étudié par Lagrange, où les corps forment un triangle équilatéral en rotation, conservant leur géométrie relative.

## 3. Modélisation Physique

### Loi de la Gravitation Universelle

Chaque corps $i$ de masse $m_i$ subit une accélération $\mathbf{a}_i$ résultant de l'attraction gravitationnelle de tous les autres corps $j$ :

$$\mathbf{a}_i = G \sum_{j \neq i} m_j \frac{\mathbf{r}_j - \mathbf{r}_i}{|\mathbf{r}_j - \mathbf{r}_i|^3}$$

où :
- $G$ est la constante gravitationnelle ($6.67430 \times 10^{-11} \text{ m}^3 \text{kg}^{-1} \text{s}^{-2}$).
- $\mathbf{r}_i$ est le vecteur position du corps $i$.

### Prévention des Singularités (Softening)

Pour éviter les accélérations infinies lors de collisions ou de rencontres très proches, une longueur de "softening" $\epsilon$ (Plummer softening) est introduite :

$$\mathbf{a}_i = G \sum_{j \neq i} m_j \frac{\mathbf{r}_j - \mathbf{r}_i}{(|\mathbf{r}_j - \mathbf{r}_i|^2 + \epsilon^2)^{3/2}}$$

### Vecteur d'État

Le système d'équations différentielles d'ordre 2 est transformé en un système d'ordre 1 en utilisant un vecteur d'état $\mathbf{y}$ regroupant positions $\mathbf{r}$ et vitesses $\mathbf{v}$ :

$$\mathbf{y} = \begin{pmatrix} \mathbf{r} \\ \mathbf{v} \end{pmatrix}, \quad \frac{d\mathbf{y}}{dt} = \begin{pmatrix} \mathbf{v} \\ \mathbf{a}(\mathbf{r}) \end{pmatrix}$$


## 3. Méthode Numérique : Intégration de Runge-Kutta (RK4)

Pour résoudre l'équation $\frac{dy}{dt} = f(y, t)$, nous utilisons l'algorithme de **Runge-Kutta d'ordre 4**. C'est un excellent compromis entre précision et coût de calcul.

Pour un pas de temps $h$ :
- $k_1 = f(y_n, t_n)$
- $k_2 = f(y_n + \frac{h}{2}k_1, t_n + \frac{h}{2})$
- $k_3 = f(y_n + \frac{h}{2}k_2, t_n + \frac{h}{2})$
- $k_4 = f(y_n + hk_3, t_n + h)$
- $y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$


## 4. Architecture Logicielle

Le projet original est décomposé en modules spécialisés. Dans ce calepin, ces composants ont été regroupés pour permettre une exécution autonome. L'organisation logique reste la suivante :

1. **Physique** (`physics`) : Fonctions de calcul des accélérations et de la dérivée du vecteur d'état.
2. **Intégration** (`integrators`) : Implémentation de l'algorithme RK4.
3. **Simulation** (`simulate`) : Orchestration de la simulation et gestion des paramètres.
4. **Visualisation** (`visualize`) : Création de l'animation interactive avec `matplotlib`.

Un lien vers le dépôt GitHub complet (contenant la structure modulaire et les tests) sera inséré ici : [Lien vers le dépôt GitHub]


## 5. Implémentation complète

Pour permettre la portabilité de ce calepin, l'ensemble du code source nécessaire est inclus ci-dessous. Cette section regroupe les fonctions de physique, l'intégrateur numérique et les outils de visualisation.

### Code Source

#### Configuration initiale

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from typing import Optional, Sequence, Callable, Tuple
from IPython.display import HTML

# --- Constantes ---
G = 6.67430e-11
DEFAULT_COLORS = ["tab:orange", "tab:blue", "tab:green", "tab:red", "tab:purple", "tab:brown"]


**Choix techniques :** Nous utilisons `numpy` pour les calculs vectorisés, ce qui est indispensable pour les performances dans un problème à N corps. Les bibliothèques de visualisation standard de `matplotlib` sont utilisées pour générer les animations.

#### Module Physique

In [ ]:
def pack_state(r: np.ndarray, v: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    return np.stack((r, v), axis=0)

def unpack_state(state: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return state[0], state[1]

def accelerations(r: np.ndarray, m: np.ndarray, softening: float = 0.0) -> np.ndarray:
    N = r.shape[0]
    a = np.zeros_like(r)
    for i in range(N):
        dr = r - r[i]
        dist2 = np.sum(dr**2, axis=1) + softening**2
        dist2[i] = np.inf
        inv_r3 = (dist2 ** -1.5)
        a[i] = G * np.sum((m[:, None] * dr) * inv_r3[:, None], axis=0)
    return a

def deriv(state: np.ndarray, m: np.ndarray, softening: float = 0.0) -> np.ndarray:
    r, v = unpack_state(state)
    a = accelerations(r, m, softening=softening)
    return np.stack((v, a), axis=0)


**Fonctionnement :** Ce bloc gère la transformation du problème en un système d'équations différentielles de premier ordre. La fonction `accelerations` implémente directement la loi de Newton avec le "softening" de Plummer. Le format de vecteur d'état `(2, N, 3)` (regroupant positions et vitesses) est choisi pour sa compatibilité directe avec les solveurs d'équations différentielles standards.

#### Module Intégrateur

In [ ]:
def rk4_step(y: np.ndarray, t: float, dt: float, f: Callable[[np.ndarray, float], np.ndarray]) -> np.ndarray:
    k1 = f(y, t)
    k2 = f(y + 0.5 * dt * k1, t + 0.5 * dt)
    k3 = f(y + 0.5 * dt * k2, t + 0.5 * dt)
    k4 = f(y + dt * k3, t + dt)
    return y + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

def integrate(y0: np.ndarray, t0: float, tf: float, dt: float, f: Callable[[np.ndarray, float], np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    n_steps = int(np.ceil((tf - t0) / dt))
    times = t0 + np.arange(n_steps + 1) * dt
    times[-1] = tf
    y = y0.copy()
    Y = np.empty((len(times),) + y0.shape, dtype=float)
    Y[0] = y
    t = t0
    for i in range(1, len(times)):
        this_dt = times[i] - t
        y = rk4_step(y, t, this_dt, f)
        t = times[i]
        Y[i] = y
    return times, Y


**Fonctionnement :** L'intégrateur implémente l'algorithme Runge-Kutta d'ordre 4 (RK4). `integrate` gère la boucle temporelle et le stockage des états successifs.

#### Module Simulation

In [ ]:
def make_rhs(masses: np.ndarray, softening: float = 0.0) -> Callable[[np.ndarray, float], np.ndarray]:
    def f(state: np.ndarray, t: float) -> np.ndarray:
        return deriv(state, masses, softening=softening)
    return f

def simulate(masses: np.ndarray, r0: np.ndarray, v0: np.ndarray, t_total: float, dt: float, softening: float = 0.0) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    state0 = pack_state(r0, v0)
    f = make_rhs(masses, softening=softening)
    times, Y = integrate(state0, 0.0, t_total, dt, f)
    return times, Y[:, 0], Y[:, 1], Y


**Fonctionnement :** Ce bloc orchestre la simulation. `make_rhs` crée une fermeture (closure) qui lie les masses fixes à la fonction dérivée, permettant à l'intégrateur de rester générique. Cette approche modulaire permet de changer d'intégrateur ou de modèle physique sans modifier la logique globale de simulation.

#### Module Visualisation

In [ ]:
def _auto_limits(R: np.ndarray, margin: float = 0.05):
    mins, maxs = R.min(axis=(0, 1)), R.max(axis=(0, 1))
    cx, cy = 0.5 * (mins[0] + maxs[0]), 0.5 * (mins[1] + maxs[1])
    rng = max(maxs[0] - mins[0], maxs[1] - mins[1]) or 1.0
    pad = margin * rng
    return (cx - 0.5 * rng - pad, cx + 0.5 * rng + pad), (cy - 0.5 * rng - pad, cy + 0.5 * rng + pad)

def animate(times, R, labels=None, colors=None, trail=None, interval_ms=20, dims=(0,1), autoscale=True):
    T, N, _ = R.shape
    if labels is None: labels = [f"Body {i+1}" for i in range(N)]
    if colors is None: colors = [DEFAULT_COLORS[i % len(DEFAULT_COLORS)] for i in range(N)]
    
    fig, ax = plt.subplots(figsize=(6, 6))
    xl, yl = _auto_limits(R)
    ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    time_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", bbox=dict(facecolor="white", alpha=0.6))
    scatters = [ax.scatter([], [], s=30, color=colors[i], label=labels[i]) for i in range(N)]
    trails = [ax.plot([], [], color=colors[i], lw=1.5, alpha=0.8)[0] for i in range(N)]
    ax.legend(loc='upper right')

    def update(frame):
        time_text.set_text(f"t = {times[frame]:,.2f} s")
        t0 = 0 if (trail is None or trail <= 0) else max(0, frame - trail)
        for i in range(N):
            trails[i].set_data(R[t0:frame+1, i, dims[0]], R[t0:frame+1, i, dims[1]])
            scatters[i].set_offsets([R[frame, i, dims[0]], R[frame, i, dims[1]]])
        if autoscale:
            xl, yl = _auto_limits(R[:frame+1])
            ax.set_xlim(*xl); ax.set_ylim(*yl)
        return [time_text, *scatters, *trails]

    return FuncAnimation(fig, update, frames=T, blit=not autoscale, interval=interval_ms)


**Fonctionnement :** Utilise `FuncAnimation` pour mettre à jour les positions des corps et leurs trajectoires (trails) à chaque itération. L'option d'autoscale permet d'adapter la vue dynamiquement. L'utilisation de `blit=not autoscale` est un choix technique pour optimiser le rendu : si les limites sont fixes, on ne redessine que les éléments mobiles, améliorant la fluidité.


## 6. Démonstration Technique

Nous allons maintenant charger un exemple concret : le système **Soleil-Terre-Saturne**.

### Configuration de l'environnement

In [ ]:
import json

# Augmenter la limite de taille pour les animations (en MB)
plt.rcParams["animation.embed_limit"] = 50.0
# Configuration pour l'affichage interactif
plt.rcParams["animation.html"] = "jshtml"


### Chargement des données d'exemple

Nous utilisons les paramètres du système **Soleil-Terre-Saturne**. Ces données sont incluses directement ici pour garantir que le calepin reste fonctionnel même sans fichiers externes.

In [ ]:
config = {
    "masses": [1.98847e30, 5.9722e24, 5.6834e26],
    "r0": [[0.0, 0.0, 0.0], [1.495978707e11, 0.0, 0.0], [1.43353e12, 0.0, 0.0]],
    "v0": [[0.0, -2.84, 0.0], [0.0, 29788.0, 0.0], [0.0, 9640.0, 0.0]],
    "t_total": 2.0e9,
    "dt": 216000.0,
    "softening": 1e9,
    "labels": ["Soleil", "Terre", "Saturne"]
}

masses = np.array(config['masses'])
r0 = np.array(config['r0'])
v0 = np.array(config['v0'])
t_total = config['t_total']
dt = config['dt']
labels = config['labels']
softening = config.get('softening', 0.0)

print(f"Simulation de {len(masses)} corps sur {t_total:.1e} secondes.")


### Exécution de la simulation

In [ ]:
times, R, V, _ = simulate(masses, r0, v0, t_total, dt, softening=softening)
print(f"Simulation terminée : {len(times)} points calculés.")


### Visualisation des résultats

L'animation suivante montre les trajectoires dans le plan XY. 

**Note sur la performance** : Pour assurer une lecture fluide dans le calepin jupyter, nous affichons un point sur 10 de la simulation (sous-échantillonnage). Cela réduit la taille du fichier d'animation tout en conservant la précision de la trajectoire calculée. Cette étape peut généralement prendre quelques dizaines de secondes pour convertir l'animation en format affichable dans jupyter.

In [ ]:
# Sous-échantillonnage pour l'animation (1 point sur 10)
step = 10
ani = animate(times[::step], R[::step], labels=labels, interval_ms=30)
HTML(ani.to_jshtml())


## 7. Conclusion et Synthèse

### Bilan du Programme

Cette modélisation numérique offre un outil flexible pour l'exploration de la mécanique céleste, tout en présentant des compromis inhérents aux choix techniques effectués.

**Points Forts :**
*   **Modularité et Clarté** : L'architecture permet une compréhension directe du passage des lois physiques aux algorithmes.
*   **Précision Locale (RK4)** : L'algorithme de Runge-Kutta d'ordre 4 offre une excellente précision pour les systèmes hiérarchiques sur des échelles de temps modérées.
*   **Robustesse Numérique** : L'usage du "softening" prévient les instabilités lors de rencontres proches.

**Points Faibles et Axes d'Amélioration :**
*   **Dérive Énergétique** : Le RK4 n'étant pas **symplectique**, l'énergie totale n'est pas conservée sur le très long terme. *Amélioration : Implémenter un intégrateur de type Yoshida ou Forest-Ruth.*
*   **Inefficacité du Pas Fixe** : Le temps de calcul est gaspillé dans les zones stables ou insuffisant lors de passages rapides. *Amélioration : Utiliser un pas adaptatif (Dormand-Prince).*
*   **Raideur Numérique** : Pour des précisions extrêmes ou des systèmes très contraints, le RK4 peut peiner. *Amélioration : Employer des méthodes implicites de haut ordre comme **Radau IIA**.*

### Réponse à la problématique : Comment trouver la stabilité ?

Trouver des conditions initiales menant à la stabilité dans un système à 3 corps, par nature chaotique, repose sur trois approches principales :

1.  **La Hiérarchie (Approche Naturelle)** : Placer les corps de manière à ce que les interactions soient dominées par un corps massif central (systèmes quasi-keplériens), ou séparer les échelles de distance (un couple serré orbité par un corps lointain).
2.  **La Symétrie et la Périodicité (Approche Théorique)** : Exploiter les solutions mathématiques particulières où les forces se compensent par la géométrie du mouvement (Points de Lagrange, Figure en 8, Triangle équilatéral).
3.  **La Recherche Numérique (Approche Computationnelle)** : Utiliser des algorithmes d'optimisation (minimisation de l'action) et des cartes de survie (indicateurs de chaos comme le Lyapunov ou le MEGNO) pour isoler les "îlots de stabilité" dans l'espace des phases.

---

L'ensemble du code ainsi que d'autres exemples sont disponibles sur le dépot git suivant : https://github.com/PaulPR-44/MNP/
Projet porté et développé par [Nicolas Fortun](https://github.com/NFortun) et [Paul Peron Redon](https://github.com/PaulPR-44) dans le cadre de l'UE [Modélisation numérique en physique](https://formations-sciences.sorbonne-universite.fr/dl/UE%20licences/UE%20licence%20physique/Fiche_UE_Mod%C3%A9lisation_num%C3%A9rique_en_physique_UL2PY222_16.05.2025.pdf) de Sorbonne Université.

Remerciements à Pacome Delva et Elena Seran pour leur accompagnement.